In [1]:
import matplotlib.pyplot as plt
import polars as pl
from polars import col as c
import pandas as pd
from src.signals.combine import make_signal

In [2]:
INPUT = '../data/processed/features.parquet'

In [3]:
lf = pl.scan_parquet(INPUT).sort(['ticker', 'date'])

In [4]:
lf = lf.filter(
    c('ticker').is_in( c('ticker').unique().sample(5).implode() )
)

In [6]:
(
    make_signal(lf, 'mom20', 'decile', scale=1)
    .filter(
        c('ticker').is_in( c('ticker').unique().sample(1).implode() )
    )
).collect().to_pandas().set_index('date')

,ticker,open,high,low,close,volume,logret,mkt_logret,beta60,beta252,...,std120,std252,dollar_volume,log_adv5,log_adv10,log_adv20,log_adv60,log_adv120,log_adv252,mom20_decile
date,,,,,,,,,,,,,,,,,,,,,
2000-01-03,GE,244.414307,245.512573,238.323914,129.276810,4605131,NaN,NaN,NaN,NaN,...,NaN,NaN,5.953366e+08,NaN,NaN,NaN,NaN,NaN,NaN,0
2000-01-04,GE,235.228806,236.426910,230.036987,124.105682,4615898,-0.040822,-0.026908,NaN,NaN,...,NaN,NaN,5.728592e+08,NaN,NaN,NaN,NaN,NaN,NaN,0
2000-01-05,GE,229.637619,234.829437,227.740616,123.890244,5694973,-0.001737,0.007533,NaN,NaN,...,NaN,NaN,7.055516e+08,NaN,NaN,NaN,NaN,NaN,NaN,0
2000-01-06,GE,228.639206,234.729584,227.840454,125.546585,4146783,0.013281,0.003879,NaN,NaN,...,NaN,NaN,5.206144e+08,NaN,NaN,NaN,NaN,NaN,NaN,0
2000-01-07,GE,236.426910,242.617142,234.829437,130.407974,4202747,0.037991,0.026325,NaN,NaN,...,NaN,NaN,5.480717e+08,20.193065,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-07-01,GE,375.029999,376.690002,371.380005,374.473206,3951400,0.003232,0.003504,2.055397,1.102599,...,0.024230,0.019936,1.479693e+09,21.553327,21.511845,21.321077,21.317106,21.311632,21.153626,1
2026-07-02,GE,379.790009,382.970001,374.369995,377.049988,3274700,0.006858,0.007797,2.037151,1.131048,...,0.024076,0.019818,1.234726e+09,21.404972,21.486125,21.319366,21.314458,21.308967,21.152229,1
2026-07-06,GE,377.170013,380.290009,373.839996,378.679993,4065400,0.004314,0.000077,1.890897,1.134536,...,0.024001,0.019782,1.539486e+09,21.235418,21.426880,21.323513,21.310392,21.310888,21.152293,1


In [ ]:
one_date = lf.filter(pl.col('date') == c('date').max()[0])
methods = ['zscore_tanh', 'zscore_clip', 'rank', 'decile']
fig, axes = plt.subplots(1, len(methods), figsize=(20, 4), sharey=True)

for ax, method in zip(axes, methods):
    signal_col = 'mom20_decile_signal' if method == 'decile' else 'mom20_signal'
    out = make_signal(one_date.lazy(), 'mom20', method=method).collect()
    ax.scatter(out['mom20'], out[signal_col], s=8, alpha=0.5)
    ax.set_title(method)
    ax.set_xlabel('raw mom20')
    ax.axhline(0, color='grey', lw=0.5)

axes[0].set_ylabel('signal')
plt.tight_layout()
plt.show()

TypeError: LazyFrame is not subscriptable (aside from slicing)

Use `select()` or `filter()` instead.